In [ ]:
# ============================================
#Phase 1: Data Cleaning
# ============================================

import pandas as pd
import numpy as np
import os
from datetime import datetime

# For visualization
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
RAW_DATA_PATH = '../data/raw/'
PROCESSED_DATA_PATH = '../data/processed/'

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

files = {
    'campaigns': 'campaigns.csv',
    'customers': 'customers.csv',
    'events': 'events.csv',
    'products': 'products.csv',
    'transactions': 'transactions.csv'
}

print("File paths configured")
print(f"Looking for data in: {RAW_DATA_PATH}")

File paths configured
Looking for data in: ../data/raw/


In [7]:
df_campaigns = pd.read_csv(os.path.join(RAW_DATA_PATH, files['campaigns']))
df_customers = pd.read_csv(os.path.join(RAW_DATA_PATH, files['customers']))
df_events = pd.read_csv(os.path.join(RAW_DATA_PATH, files['events']))
df_products = pd.read_csv(os.path.join(RAW_DATA_PATH, files['products']))
df_transactions = pd.read_csv(os.path.join(RAW_DATA_PATH, files['transactions']))

print("=" * 50)
print("CAMPAIGNS TABLE")
print("=" * 50)
print(f"Shape: {df_campaigns.shape}")
print(df_campaigns.head(2))

print("\n" + "=" * 50)
print("CUSTOMERS TABLE")
print("=" * 50)
print(f"Shape: {df_customers.shape}")
print(df_customers.head(2))

print("\n" + "=" * 50)
print("EVENTS TABLE")
print("=" * 50)
print(f"Shape: {df_events.shape}")
print(df_events.head(2))

print("\n" + "=" * 50)
print("PRODUCTS TABLE")
print("=" * 50)
print(f"Shape: {df_products.shape}")
print(df_products.head(2))

print("\n" + "=" * 50)
print("TRANSACTIONS TABLE")
print("=" * 50)
print(f"Shape: {df_transactions.shape}")
print(df_transactions.head(2))

CAMPAIGNS TABLE
Shape: (50, 7)
   campaign_id      channel   objective  start_date    end_date  \
0            1  Paid Search  Cross-sell  2021-10-25  2021-11-26   
1            2        Email   Retention  2021-10-24  2021-12-24   

  target_segment  expected_uplift  
0   Deal Seekers            0.022  
1   Deal Seekers            0.116  

CUSTOMERS TABLE
Shape: (100000, 7)
   customer_id signup_date country  age  gender loyalty_tier  \
0            1  2021-04-08      BR   48    Male       Bronze   
1            2  2023-04-28      IN   36  Female       Silver   

  acquisition_channel  
0            Referral  
1             Organic  

EVENTS TABLE
Shape: (2000000, 12)
   event_id            timestamp  customer_id  session_id   event_type  \
0         1  2021-01-14 13:35:43        43812      535101         view   
1         2  2021-12-03 21:36:50        71340       96426  add_to_cart   

   product_id device_type traffic_source  campaign_id page_category  \
0      1004.0     desktop    

In [ ]:
def check_data_quality(df, name):
    print(f"\n--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Missing values:\n{df.isnull().sum()}")
    print(f"\nData types:\n{df.dtypes}")
    print("-" * 40)

check_data_quality(df_campaigns, "Campaigns")
check_data_quality(df_customers, "Customers")
check_data_quality(df_events, "Events")
check_data_quality(df_products, "Products")
check_data_quality(df_transactions, "Transactions")


--- Campaigns ---
Shape: (50, 7)
Missing values:
campaign_id        0
channel            0
objective          0
start_date         0
end_date           0
target_segment     0
expected_uplift    0
dtype: int64

Data types:
campaign_id          int64
channel                str
objective              str
start_date             str
end_date               str
target_segment         str
expected_uplift    float64
dtype: object
----------------------------------------

--- Customers ---
Shape: (100000, 7)
Missing values:
customer_id            0
signup_date            0
country                0
age                    0
gender                 0
loyalty_tier           0
acquisition_channel    0
dtype: int64

Data types:
customer_id            int64
signup_date              str
country                  str
age                    int64
gender                   str
loyalty_tier             str
acquisition_channel      str
dtype: object
----------------------------------------

--- Events ---
Shap

In [ ]:
print("Cleaning Campaigns...")

if 'start_date' in df_campaigns.columns:
    df_campaigns['start_date'] = pd.to_datetime(df_campaigns['start_date'])
if 'end_date' in df_campaigns.columns:
    df_campaigns['end_date'] = pd.to_datetime(df_campaigns['end_date'])

print(f"Campaigns shape after cleaning: {df_campaigns.shape}")
print(df_campaigns.dtypes)

Cleaning Campaigns...
Campaigns shape after cleaning: (50, 7)
campaign_id                 int64
channel                       str
objective                     str
start_date         datetime64[us]
end_date           datetime64[us]
target_segment                str
expected_uplift           float64
dtype: object


In [ ]:
print("\nCleaning Customers...")

print(f"Missing values before: {df_customers.isnull().sum().sum()}")

if 'loyalty_tier' in df_customers.columns:
    df_customers['loyalty_tier'] = df_customers['loyalty_tier'].fillna('Bronze')

if 'traffic_source' in df_customers.columns:
    df_customers['traffic_source'] = df_customers['traffic_source'].fillna('Unknown')

df_customers = df_customers.dropna(subset=['customer_id'])

print(f"Shape after cleaning: {df_customers.shape}")
print(f"Missing values after: {df_customers.isnull().sum().sum()}")


Cleaning Customers...
Missing values before: 0
Shape after cleaning: (100000, 7)
Missing values after: 0


In [12]:
print("\nCleaning Events...")

print(f"Available columns: {df_events.columns.tolist()}")

if 'experiment_group' in df_events.columns:
    print(f"Unique experiment groups: {df_events['experiment_group'].unique()}")
    
    df_events['experiment_group'] = df_events['experiment_group'].fillna('unknown')

if 'timestamp' in df_events.columns:
    df_events['timestamp'] = pd.to_datetime(df_events['timestamp'])
    
    df_events['event_date'] = df_events['timestamp'].dt.date
    df_events['event_hour'] = df_events['timestamp'].dt.hour
    df_events['event_dayofweek'] = df_events['timestamp'].dt.dayofweek
    df_events['event_month'] = df_events['timestamp'].dt.month
    
    print(f"Timestamp range: {df_events['timestamp'].min()} to {df_events['timestamp'].max()}")

if 'session_id' in df_events.columns:
    df_events['session_id'] = df_events['session_id'].fillna('unknown')

if 'device_type' in df_events.columns:
    df_events['device_type'] = df_events['device_type'].fillna('unknown')
    print(f"Device types: {df_events['device_type'].unique()}")

if 'traffic_source' in df_events.columns:
    df_events['traffic_source'] = df_events['traffic_source'].fillna('direct')

if 'page_category' in df_events.columns:
    df_events['page_category'] = df_events['page_category'].fillna('other')

if 'customer_id' in df_events.columns and 'timestamp' in df_events.columns and 'event_type' in df_events.columns:
    before_dedup = len(df_events)
    df_events = df_events.drop_duplicates(subset=['customer_id', 'timestamp', 'event_type'])
    after_dedup = len(df_events)
    print(f"Removed {before_dedup - after_dedup} duplicate events")

print(f"Shape after cleaning: {df_events.shape}")
print(f"Missing values after cleaning: {df_events.isnull().sum().sum()}")


Cleaning Events...
Available columns: ['event_id', 'timestamp', 'customer_id', 'session_id', 'event_type', 'product_id', 'device_type', 'traffic_source', 'campaign_id', 'page_category', 'session_duration_sec', 'experiment_group']
Unique experiment groups: <ArrowStringArray>
['Control', 'Variant_A', 'Variant_B']
Length: 3, dtype: str
Timestamp range: 2021-01-01 00:01:28 to 2023-12-31 23:57:50
Device types: <ArrowStringArray>
['desktop', 'mobile', 'tablet', 'unknown']
Length: 4, dtype: str
Removed 0 duplicate events
Shape after cleaning: (2000000, 16)
Missing values after cleaning: 200371


In [14]:
print("\nCleaning Products...")

if 'category' in df_products.columns:
    df_products['category'] = df_products['category'].fillna('Uncategorized')

if 'price' in df_products.columns:
    median_price = df_products['price'].median()
    df_products['price'] = df_products['price'].fillna(median_price)
    print(f"Filled missing prices with median: ${median_price:.2f}")

print(f"Shape after cleaning: {df_products.shape}")


Cleaning Products...
Shape after cleaning: (2000, 6)


In [16]:
print("\nCleaning Transactions...")

if 'transaction_date' in df_transactions.columns:
    df_transactions['transaction_date'] = pd.to_datetime(df_transactions['transaction_date'])

if 'revenue' in df_transactions.columns:
    df_transactions['revenue'] = df_transactions['revenue'].fillna(0)

if 'refund_flag' in df_transactions.columns:
    df_transactions['refund_flag'] = df_transactions['refund_flag'].fillna(False)

df_transactions = df_transactions.dropna(subset=['transaction_id'])

print(f"Shape after cleaning: {df_transactions.shape}")


Cleaning Transactions...
Shape after cleaning: (103127, 9)


In [ ]:
print("Merging tables for A/B test analysis...")
print("=" * 60)

print("\nAVAILABLE TABLES & COLUMNS:")
print(f"Events: {df_events.columns.tolist()}")
print(f"Customers: {df_customers.columns.tolist()}")
print(f"Transactions: {df_transactions.columns.tolist()}")
print(f"Products: {df_products.columns.tolist()}")
print(f"Campaigns: {df_campaigns.columns.tolist()}")
print("=" * 60)

master_df = df_events.copy()
print(f"\nStep 1 - Events table: {master_df.shape}")

customer_cols = ['customer_id', 'signup_date', 'country', 'age', 'gender', 'loyalty_tier', 'acquisition_channel']
existing_customer_cols = [col for col in customer_cols if col in df_customers.columns]
master_df = master_df.merge(
    df_customers[existing_customer_cols], 
    on='customer_id', 
    how='left'
)
print(f"Step 2 - After customers join: {master_df.shape}")

if 'transaction_id' in df_transactions.columns:
    transactions_clean = df_transactions.copy()
    
    if 'timestamp' in transactions_clean.columns:
        transactions_clean['timestamp'] = pd.to_datetime(transactions_clean['timestamp'])
    
    transactions_clean['has_transaction'] = 1
    
    if 'refund_flag' in transactions_clean.columns:
        transactions_clean['refund_flag'] = transactions_clean['refund_flag'].fillna(False)
    
    if 'gross_revenue' in transactions_clean.columns:
        transactions_clean['gross_revenue'] = transactions_clean['gross_revenue'].fillna(0)
        if 'refund_flag' in transactions_clean.columns:
            transactions_clean['net_revenue'] = transactions_clean.apply(
                lambda x: x['gross_revenue'] if not x['refund_flag'] else 0, axis=1
            )
        else:
            transactions_clean['net_revenue'] = transactions_clean['gross_revenue']
    
    master_df = master_df.merge(
        transactions_clean[['transaction_id', 'customer_id', 'timestamp', 'product_id', 
                           'quantity', 'discount_applied', 'gross_revenue', 'net_revenue', 
                           'refund_flag', 'campaign_id']],
        on='customer_id',
        how='left',
        suffixes=('', '_txn')
    )
    print(f"Step 3 - After transactions join: {master_df.shape}")

if 'product_id' in master_df.columns and 'product_id' in df_products.columns:
    product_cols = ['product_id', 'product_name', 'category', 'subcategory', 'price', 'brand']
    existing_product_cols = [col for col in product_cols if col in df_products.columns]
    master_df = master_df.merge(
        df_products[existing_product_cols],
        on='product_id',
        how='left'
    )
    print(f"Step 4 - After products join: {master_df.shape}")

if 'campaign_id' in master_df.columns and 'campaign_id' in df_campaigns.columns:
    campaign_cols = ['campaign_id', 'campaign_name', 'channel', 'start_date', 'end_date', 'objective']
    existing_campaign_cols = [col for col in campaign_cols if col in df_campaigns.columns]
    master_df = master_df.merge(
        df_campaigns[existing_campaign_cols],
        on='campaign_id',
        how='left'
    )
    print(f"Step 5 - After campaigns join: {master_df.shape}")

print("=" * 60)
print(f"Final master table shape: {master_df.shape}")
print(f"Columns in master table: {len(master_df.columns)}")

print("\nPreview of master table:")
print(master_df.head(3))

Merging tables for A/B test analysis...

📊 AVAILABLE TABLES & COLUMNS:
Events: ['event_id', 'timestamp', 'customer_id', 'session_id', 'event_type', 'product_id', 'device_type', 'traffic_source', 'campaign_id', 'page_category', 'session_duration_sec', 'experiment_group', 'event_date', 'event_hour', 'event_dayofweek', 'event_month']
Customers: ['customer_id', 'signup_date', 'country', 'age', 'gender', 'loyalty_tier', 'acquisition_channel']
Transactions: ['transaction_id', 'timestamp', 'customer_id', 'product_id', 'quantity', 'discount_applied', 'gross_revenue', 'campaign_id', 'refund_flag']
Products: ['product_id', 'category', 'brand', 'base_price', 'launch_date', 'is_premium']
Campaigns: ['campaign_id', 'channel', 'objective', 'start_date', 'end_date', 'target_segment', 'expected_uplift']

Step 1 - Events table: (2000000, 16)
Step 2 - After customers join: (2000000, 22)
Step 3 - After transactions join: (2848119, 31)
Step 4 - After products join: (2848119, 33)
Step 5 - After campaigns j

In [ ]:

print("\n" + "=" * 60)
print("CREATING A/B TEST METRICS")
print("=" * 60)

if 'transaction_id' in master_df.columns:
    master_df['converted'] = master_df['transaction_id'].notna().astype(int)
    print(f"Overall conversion rate: {master_df['converted'].mean():.2%}")

if 'net_revenue' in master_df.columns:
    master_df['revenue'] = master_df['net_revenue'].fillna(0)
elif 'gross_revenue' in master_df.columns:
    master_df['revenue'] = master_df['gross_revenue'].fillna(0)
else:
    master_df['revenue'] = 0
    print("No revenue column found - creating placeholder")

if 'experiment_group' in master_df.columns:
    master_df['is_treatment'] = (master_df['experiment_group'] == 'treatment').astype(int)
    master_df['is_control'] = (master_df['experiment_group'] == 'control').astype(int)
    
    print(f"\nGroup distribution (event-level):")
    print(master_df['experiment_group'].value_counts())
    
    other_groups = master_df[~master_df['experiment_group'].isin(['control', 'treatment'])]['experiment_group'].unique()
    if len(other_groups) > 0:
        print(f"Other groups found: {other_groups}")
else:
    print("No experiment_group column found!")

if 'timestamp' in master_df.columns:
    master_df['timestamp'] = pd.to_datetime(master_df['timestamp'])
    master_df['event_date'] = master_df['timestamp'].dt.date
    master_df['event_hour'] = master_df['timestamp'].dt.hour
    master_df['event_dayofweek'] = master_df['timestamp'].dt.dayofweek  # Monday=0, Sunday=6
    master_df['event_weekend'] = (master_df['event_dayofweek'] >= 5).astype(int)
    master_df['event_month'] = master_df['timestamp'].dt.month
    print(f"Time features extracted (range: {master_df['timestamp'].min()} to {master_df['timestamp'].max()})")

if 'signup_date' in master_df.columns:
    master_df['signup_date'] = pd.to_datetime(master_df['signup_date'])
    if 'timestamp' in master_df.columns:
        master_df['customer_tenure_days'] = (master_df['timestamp'] - master_df['signup_date']).dt.days
        print(f"Customer tenure calculated (avg: {master_df['customer_tenure_days'].mean():.0f} days)")

print("\n" + "=" * 60)
print("Derived columns created successfully")


CREATING A/B TEST METRICS
✅ Overall conversion rate: 76.01%

📊 Group distribution (event-level):
experiment_group
Control      1703606
Variant_B     572501
Variant_A     572012
Name: count, dtype: int64
⚠️ Other groups found: <ArrowStringArray>
['Control', 'Variant_A', 'Variant_B']
Length: 3, dtype: str
✅ Time features extracted (range: 2021-01-01 00:01:28 to 2023-12-31 23:57:50)
✅ Customer tenure calculated (avg: 1 days)

✅ Derived columns created successfully


In [ ]:

print("\n" + "=" * 60)
print("CREATING USER-LEVEL ANALYSIS TABLE")
print("=" * 60)

agg_dict = {}

if 'is_treatment' in master_df.columns:
    agg_dict['is_treatment'] = 'first'
if 'is_control' in master_df.columns:
    agg_dict['is_control'] = 'first'
if 'experiment_group' in master_df.columns:
    agg_dict['experiment_group'] = 'first'

if 'converted' in master_df.columns:
    agg_dict['converted'] = 'max'

if 'revenue' in master_df.columns:
    agg_dict['revenue'] = 'sum'
if 'gross_revenue' in master_df.columns:
    agg_dict['gross_revenue'] = 'sum'
if 'net_revenue' in master_df.columns:
    agg_dict['net_revenue'] = 'sum'

if 'quantity' in master_df.columns:
    agg_dict['quantity'] = 'sum'

if 'discount_applied' in master_df.columns:
    agg_dict['discount_applied'] = 'sum'

customer_attrs = ['loyalty_tier', 'country', 'age', 'gender', 'acquisition_channel', 'signup_date']
for attr in customer_attrs:
    if attr in master_df.columns:
        agg_dict[attr] = 'first'

if 'session_id' in master_df.columns:
    agg_dict['session_id'] = 'nunique'  
if 'event_type' in master_df.columns:
    agg_dict['event_type'] = 'count'  

if 'device_type' in master_df.columns:
    agg_dict['device_type'] = lambda x: x.mode()[0] if len(x.mode()) > 0 else 'unknown'
if 'traffic_source' in master_df.columns:
    agg_dict['traffic_source'] = lambda x: x.mode()[0] if len(x.mode()) > 0 else 'unknown'

if 'timestamp' in master_df.columns:
    agg_dict['first_event_time'] = ('timestamp', 'min')
    agg_dict['last_event_time'] = ('timestamp', 'max')

if 'transaction_id' in master_df.columns:
    agg_dict['transaction_count'] = ('transaction_id', 'count')

try:
    user_level_df = master_df.groupby('customer_id').agg(agg_dict).reset_index()
    print(f"User-level table created: {user_level_df.shape}")
except Exception as e:
    print(f"Error with agg_dict: {e}")
    print("Trying simpler aggregation...")
    user_level_df = master_df.groupby('customer_id').agg({
        'experiment_group': 'first',
        'converted': 'max',
        'revenue': 'sum',
        'loyalty_tier': 'first',
        'device_type': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'unknown'
    }).reset_index()
    print(f"User-level table created (simplified): {user_level_df.shape}")

if 'experiment_group' in user_level_df.columns:
    print(f"\nUser-level group distribution:")
    print(user_level_df['experiment_group'].value_counts())
    
    n_control = (user_level_df['experiment_group'] == 'control').sum()
    n_treatment = (user_level_df['experiment_group'] == 'treatment').sum()
    print(f"Control users: {n_control:,}")
    print(f"Treatment users: {n_treatment:,}")
    
    if 'converted' in user_level_df.columns:
        conv_control = user_level_df[user_level_df['experiment_group'] == 'control']['converted'].mean()
        conv_treatment = user_level_df[user_level_df['experiment_group'] == 'treatment']['converted'].mean()
        print(f"\nUser-level conversion rates:")
        print(f"Control: {conv_control:.2%}")
        print(f"Treatment: {conv_treatment:.2%}")
        print(f"Lift: {(conv_treatment/conv_control - 1):.2%}" if conv_control > 0 else "Lift: N/A")


CREATING USER-LEVEL ANALYSIS TABLE
Error with agg_dict: "Label(s) ['first_event_time', 'last_event_time', 'transaction_count'] do not exist"
Trying simpler aggregation...
✅ User-level table created (simplified): (100000, 6)

📊 User-level group distribution:
experiment_group
Control      60015
Variant_A    20067
Variant_B    19918
Name: count, dtype: int64
Control users: 0
Treatment users: 0

📈 User-level conversion rates:
Control: nan%
Treatment: nan%
Lift: N/A


In [22]:
# Save cleaned and processed data

print("\n" + "=" * 60)
print("SAVING PROCESSED DATA")
print("=" * 60)

# Save master event-level table
master_df.to_csv(os.path.join(PROCESSED_DATA_PATH, 'master_events_cleaned.csv'), index=False)
print(f"✅ Saved master event-level table: {master_df.shape}")

# Save user-level analysis table (main for A/B testing)
user_level_df.to_csv(os.path.join(PROCESSED_DATA_PATH, 'user_level_ab_data.csv'), index=False)
print(f"✅ Saved user-level A/B test data: {user_level_df.shape}")

# Save summary statistics
summary_stats = {
    'total_events': len(master_df),
    'unique_customers': user_level_df['customer_id'].nunique(),
    'control_users': n_control if 'n_control' in dir() else 0,
    'treatment_users': n_treatment if 'n_treatment' in dir() else 0,
    'control_conversion': conv_control if 'conv_control' in dir() else 0,
    'treatment_conversion': conv_treatment if 'conv_treatment' in dir() else 0,
}

summary_df = pd.DataFrame([summary_stats])
summary_df.to_csv(os.path.join(PROCESSED_DATA_PATH, 'summary_stats.csv'), index=False)
print(f"✅ Saved summary statistics")

print("\n✅ All files saved to: data/processed/")


SAVING PROCESSED DATA
✅ Saved master event-level table: (2848119, 43)
✅ Saved user-level A/B test data: (100000, 6)
✅ Saved summary statistics

✅ All files saved to: data/processed/


In [23]:
# Create derived columns specifically for A/B testing

# 1. Conversion flag (did they purchase?)
# Note: 'converted' might not exist directly - we need to check
if 'event_type' in master_df.columns:
    # If event_type has 'purchase' or 'transaction' events
    master_df['converted'] = (master_df['event_type'].str.contains('purchase|transaction|order', case=False, na=False)).astype(int)
    print(f"Conversion rate (overall): {master_df['converted'].mean():.2%}")
elif 'transaction_id' in master_df.columns:
    master_df['converted'] = master_df['transaction_id'].notna().astype(int)
    print(f"Conversion rate (overall): {master_df['converted'].mean():.2%}")
else:
    print("Warning: No conversion column found. Will create placeholder.")
    master_df['converted'] = 0

# 2. Revenue per user (fill non-purchasers with 0)
if 'revenue' in master_df.columns:
    master_df['revenue'] = master_df['revenue'].fillna(0)

# 3. Create control/treatment flags
if 'experiment_group' in master_df.columns:
    master_df['is_treatment'] = (master_df['experiment_group'] == 'treatment').astype(int)
    master_df['is_control'] = (master_df['experiment_group'] == 'control').astype(int)
    
    print(f"\nGroup distribution:")
    print(master_df['experiment_group'].value_counts())
else:
    print("Warning: No experiment_group column found!")
    print(f"Available columns: {master_df.columns.tolist()}")

# 4. Create user-level aggregated table (for cleaner A/B analysis)
# Use appropriate column names
group_cols = ['customer_id']
if 'experiment_group' in master_df.columns:
    group_cols.append('experiment_group')

# For aggregation, use only columns that exist
agg_dict = {}

if 'is_treatment' in master_df.columns:
    agg_dict['is_treatment'] = 'first'
if 'is_control' in master_df.columns:
    agg_dict['is_control'] = 'first'
if 'converted' in master_df.columns:
    agg_dict['converted'] = 'max'
if 'revenue' in master_df.columns:
    agg_dict['revenue'] = 'sum'
if 'loyalty_tier' in master_df.columns:
    agg_dict['loyalty_tier'] = 'first'
if 'traffic_source' in master_df.columns:
    agg_dict['traffic_source'] = 'first'
if 'device_type' in master_df.columns:
    agg_dict['device_type'] = 'first'

# Always include timestamp count if available
if 'timestamp' in master_df.columns:
    master_df['timestamp_for_count'] = master_df['timestamp']
    agg_dict['timestamp_for_count'] = 'count'

user_level_df = master_df.groupby('customer_id').agg(agg_dict).reset_index()

# Rename the count column
if 'timestamp_for_count' in user_level_df.columns:
    user_level_df = user_level_df.rename(columns={'timestamp_for_count': 'total_events'})

# Add experiment_group back if it exists in original but not in agg
if 'experiment_group' in master_df.columns and 'experiment_group' not in user_level_df.columns:
    experiment_map = master_df[['customer_id', 'experiment_group']].drop_duplicates()
    user_level_df = user_level_df.merge(experiment_map, on='customer_id', how='left')

print(f"\nUser-level analysis table shape: {user_level_df.shape}")
if 'experiment_group' in user_level_df.columns:
    print(f"Users in control: {(user_level_df['experiment_group'] == 'control').sum()}")
    print(f"Users in treatment: {(user_level_df['experiment_group'] == 'treatment').sum()}")

Conversion rate (overall): 7.45%

Group distribution:
experiment_group
Control      1703606
Variant_B     572501
Variant_A     572012
Name: count, dtype: int64

User-level analysis table shape: (296315, 10)
Users in control: 0
Users in treatment: 0


In [24]:
# Validate the data is ready for statistical analysis

print("=" * 60)
print("FINAL DATA VALIDATION")
print("=" * 60)

# Check 1: Are both groups present?
if 'experiment_group' in user_level_df.columns:
    control_exists = (user_level_df['experiment_group'] == 'control').any()
    treatment_exists = (user_level_df['experiment_group'] == 'treatment').any()
    print(f"✅ Control group present: {control_exists}")
    print(f"✅ Treatment group present: {treatment_exists}")

    # Check 2: Sufficient sample size?
    n_control = (user_level_df['experiment_group'] == 'control').sum()
    n_treatment = (user_level_df['experiment_group'] == 'treatment').sum()
    print(f"✅ Control group size: {n_control:,}")
    print(f"✅ Treatment group size: {n_treatment:,}")
else:
    print("⚠️ No experiment_group column found in user_level_df")
    n_control = 0
    n_treatment = 0

# Check 3: Conversion metrics available
if 'converted' in user_level_df.columns:
    if n_control > 0:
        conversion_control = user_level_df[user_level_df['experiment_group'] == 'control']['converted'].mean()
        print(f"✅ Control conversion: {conversion_control:.2%}")
    if n_treatment > 0:
        conversion_treatment = user_level_df[user_level_df['experiment_group'] == 'treatment']['converted'].mean()
        print(f"✅ Treatment conversion: {conversion_treatment:.2%}")
else:
    print("⚠️ No conversion column found")

# Check 4: Revenue data available
if 'revenue' in user_level_df.columns:
    if n_control > 0:
        revenue_control = user_level_df[user_level_df['experiment_group'] == 'control']['revenue'].mean()
        print(f"✅ Control avg revenue: ${revenue_control:.2f}")
    if n_treatment > 0:
        revenue_treatment = user_level_df[user_level_df['experiment_group'] == 'treatment']['revenue'].mean()
        print(f"✅ Treatment avg revenue: ${revenue_treatment:.2f}")
else:
    print("⚠️ No revenue column found")

print("\n" + "=" * 60)
print("✅ DATA VALIDATION COMPLETE")
print("=" * 60)

FINAL DATA VALIDATION
✅ Control group present: False
✅ Treatment group present: False
✅ Control group size: 0
✅ Treatment group size: 0

✅ DATA VALIDATION COMPLETE


In [27]:
# Fix group naming and calculate correct conversion rates

print("=" * 50)
print("FIXING GROUP NAMES")
print("=" * 50)

# Load the data
user_df = pd.read_csv('../data/processed/user_level_ab_data.csv')

# Show original distribution
print(f"\nOriginal group distribution:")
print(user_df['experiment_group'].value_counts())

# Standardize group names (lowercase for consistency)
user_df['experiment_group'] = user_df['experiment_group'].str.lower()

print(f"\nStandardized group distribution:")
print(user_df['experiment_group'].value_counts())

# Create simplified treatment flag
user_df['is_treatment'] = (user_df['experiment_group'] != 'control').astype(int)
user_df['is_control'] = (user_df['experiment_group'] == 'control').astype(int)

print(f"\nTreatment flag distribution:")
print(user_df['is_treatment'].value_counts())

# Calculate conversion rates correctly
print("\n" + "=" * 50)
print("CONVERSION RATES BY GROUP")
print("=" * 50)

for group in user_df['experiment_group'].unique():
    subset = user_df[user_df['experiment_group'] == group]
    conv_rate = subset['converted'].mean()
    n_users = len(subset)
    print(f"{group.upper()}: {n_users:,} users | Conversion: {conv_rate:.2%}")

# Control vs any treatment (combined)
control_conv = user_df[user_df['experiment_group'] == 'control']['converted'].mean()
treatment_conv = user_df[user_df['experiment_group'] != 'control']['converted'].mean()

print("\n" + "=" * 50)
print("CONTROL VS TREATMENT (COMBINED)")
print("=" * 50)
print(f"Control (Group 1): {control_conv:.2%}")
print(f"Treatment (Variant A + B): {treatment_conv:.2%}")
print(f"Lift: {(treatment_conv/control_conv - 1):.2%}")

# Save the fixed data
user_df.to_csv('../data/processed/user_level_ab_data_fixed.csv', index=False)
print("\n✅ Fixed data saved to: user_level_ab_data_fixed.csv")

FIXING GROUP NAMES

Original group distribution:
experiment_group
Control      60015
Variant_A    20067
Variant_B    19918
Name: count, dtype: int64

Standardized group distribution:
experiment_group
control      60015
variant_a    20067
variant_b    19918
Name: count, dtype: int64

Treatment flag distribution:
is_treatment
0    60015
1    39985
Name: count, dtype: int64

CONVERSION RATES BY GROUP
CONTROL: 60,015 users | Conversion: 63.95%
VARIANT_B: 19,918 users | Conversion: 64.08%
VARIANT_A: 20,067 users | Conversion: 64.24%

CONTROL VS TREATMENT (COMBINED)
Control (Group 1): 63.95%
Treatment (Variant A + B): 64.16%
Lift: 0.33%

✅ Fixed data saved to: user_level_ab_data_fixed.csv


In [28]:
# Check individual variant performance
print("=" * 50)
print("VARIANT BREAKDOWN")
print("=" * 50)

for group in ['control', 'variant_a', 'variant_b']:
    subset = user_df[user_df['experiment_group'] == group]
    conv_rate = subset['converted'].mean()
    n_users = len(subset)
    revenue_per_user = subset['revenue'].mean()
    print(f"\n{group.upper()}:")
    print(f"  Users: {n_users:,}")
    print(f"  Conversion: {conv_rate:.2%}")
    print(f"  Revenue/user: ${revenue_per_user:.2f}")

# Compare variants to control
control_conv = user_df[user_df['experiment_group'] == 'control']['converted'].mean()

for variant in ['variant_a', 'variant_b']:
    var_conv = user_df[user_df['experiment_group'] == variant]['converted'].mean()
    lift = (var_conv / control_conv - 1)
    print(f"\n{variant.upper()} vs CONTROL:")
    print(f"  Lift: {lift:.4%}")

VARIANT BREAKDOWN

CONTROL:
  Users: 60,015
  Conversion: 63.95%
  Revenue/user: $1801.54

VARIANT_A:
  Users: 20,067
  Conversion: 64.24%
  Revenue/user: $1822.19

VARIANT_B:
  Users: 19,918
  Conversion: 64.08%
  Revenue/user: $1834.80

VARIANT_A vs CONTROL:
  Lift: 0.4521%

VARIANT_B vs CONTROL:
  Lift: 0.2065%
